# Dicts, sets, tuples: data structure patterns

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~45 min

## The four containers, honestly

`list` — ordered, mutable, allows duplicates: the default sequence. `tuple` — the same but immutable: safe dict keys, multi-value returns. `dict` — key → value lookup: index anything by anything. `set` — unique items with instant membership tests: dedupe and "have I seen this?".

Rule of thumb: **list** for data you walk through in order, **dict** for lookup by name, **set** for membership and dedupe, **tuple** when the contents must not change — or must serve as a dict key.



In [ ]:
label2id = {"cat": 0, "dog": 1}      # dict
label2id["bird"] = 2                # add a key
label2id.get("fox", -1)             # -1 — .get returns a default instead of KeyError
"cat" in label2id                   # True — "in" on a dict checks KEYS
pair = (0.91, "cat")                # tuple — e.g. a (score, label) returned together

## Dict patterns: counting and grouping

Two dict idioms appear in every dataset pipeline. **Counting**: `d[k] = d.get(k, 0) + 1` — `.get(key, default)` avoids the KeyError on first sighting; called with no default, `d.get(k)` returns `None`. **Grouping**: collect items into per-key lists with `setdefault` — `d.setdefault(k, v)` returns `d[k]`, inserting `v` first if `k` is missing.



In [ ]:
labels = ["cat", "dog", "cat", "bird", "dog", "cat"]

counts = {}
for lab in labels:
    counts[lab] = counts.get(lab, 0) + 1
counts                          # {'cat': 3, 'dog': 2, 'bird': 1}

by_first = {}                   # group words by first letter
for w in ["cat", "dog", "crow"]:
    by_first.setdefault(w[0], []).append(w)
by_first                        # {'c': ['cat', 'crow'], 'd': ['dog']}

`collections.Counter` does the counting for you (`Counter(labels).most_common(3)`) and `defaultdict(list)` removes the setdefault — both appear in the next lesson. Write the plain version by hand first so you know what they hide.

Looping over a dict yields its **keys**; `.items()` yields (key, value) pairs; `.values()` yields values. Invert a dict — swap keys and values — with a comprehension: `{v: k for k, v in d.items()}`.



## Tuples and unpacking

Unpacking splits a sequence into names — in assignments, in `for` targets, in `return`. It is why `a, b = b, a` swaps with no temporary variable: the right side is fully built before assignment happens.



In [ ]:
def top_two(scores):
    return max(scores), min(scores)      # returns ONE tuple

hi, lo = top_two([0.4, 0.9, 0.7])        # unpacked at the call site
a, b = 1, 2
a, b = b, a                              # swap

for i, (name, sc) in enumerate([("cat", 0.9), ("dog", 0.7)]):
    print(i, name, sc)                   # nested unpacking in the for-target

Use `_` for values you don't need (`name, _ = pair`), and `first, *rest = xs` to grab the head plus everything else — `*rest` collects the remainder into a list.



## Sets: membership and dedupe

`set` stores each item once and answers `x in s` in ~O(1) — a hash lookup, ~instant regardless of the set's size — where a list scans all n items. Deduplicating 10⁶ items against a set is instant; against a list it is roughly 10¹² comparisons: seconds versus hours.



In [ ]:
seen = {"cat", "dog"}
"cat" in seen                        # True — constant time
seen.add("bird"); seen.add("cat")    # 'cat' was already there — sets keep one copy

a, b = {1, 2, 3}, {2, 3, 4}
a | b                                # {1, 2, 3, 4}  union
a & b                                # {2, 3}        intersection
a - b                                # {1}           difference
list({3, 1, 2, 1})                   # dedupe — but order is not guaranteed

- Set items and dict keys must be hashable: numbers, strings, tuples work — lists and dicts don't.
- `{1}` is a set but `{}` is an empty **dict** — write `set()` for an empty set.
- Sets are unordered: `sorted(s)` when order matters, never rely on iteration order.



## Sorting with keys

`sorted(xs, key=f)` orders by `f(x)` — the value each item maps to. A lambda picks out a field; `reverse=True` flips the order. A tuple key sorts lexicographically: first element decides, ties fall to the second.



In [ ]:
pairs = [("cat", 0.9), ("dog", 0.7), ("bird", 0.9)]
sorted(pairs, key=lambda p: p[1])            # by score, ascending
sorted(pairs, key=lambda p: -p[1])           # descending, via negation
sorted(pairs, key=lambda p: (-p[1], p[0]))   # score desc, name asc on ties

counts = {"cat": 3, "dog": 2}
sorted(counts, key=counts.get)               # ['dog', 'cat'] — dict keys by value

> **Note.** `sorted` returns a new list and leaves the original alone; `xs.sort()` mutates `xs` and returns `None`. Both accept `key=` and `reverse=`.



## Aliasing: the bug that bites everyone

`b = a` never copies a container — both names point to **the same object**, so `b.append(3)` shows up in `a` too. Copy with `a.copy()` (one level) or `copy.deepcopy(a)` (nested containers).



In [ ]:
a = [1, 2]
b = a          # b is the SAME list
b.append(3)
a              # [1, 2, 3] — surprise

c = a.copy()   # a real copy
c.append(4)    # a is untouched

d = [[0] * 3] * 2        # looks like two rows...
d[0][1] = 9              # ...but d[1][1] is also 9 — both rows are the same list
d = [[0] * 3 for _ in range(2)]   # fix: build each row separately

- `def f(xs=[])` shares one default list across all calls — mutate it once and it leaks into every later call. Default to `xs=None`.
- Slices copy shallowly: `a[:]` copies a flat list, but a list of lists still shares the inner lists — `copy.deepcopy` fixes that.
- Functions receive the same object too: a helper that mutates its argument changes the caller's data. Copy at the boundary, or document the mutation.



### ✏️ What does `{k: v for k, v in zip(['a', 'b'], [1, 2])}` produce?

**A.** An error
**B.** {'a': 1, 'b': 2}
**C.** [('a', 1), ('b', 2)]
**D.** {'a', 'b', 1, 2}

<details><summary>Answer</summary>

**B.** {'a': 1, 'b': 2}

zip pairs the items; the dict comprehension turns each pair into key: value.

</details>

### ✏️ d = {'a': 1}. What does d.get('b') return?

**A.** 'b'
**B.** 0
**C.** None
**D.** KeyError

<details><summary>Answer</summary>

**C.** None

.get() returns None (or a default you pass) instead of raising; d['b'] would raise KeyError.

</details>

### ✏️ `a = [1, 2]; b = a; b.append(3)`. What is `a` now?

**A.** [1, 2]
**B.** An error
**C.** [3]
**D.** [1, 2, 3]

<details><summary>Answer</summary>

**D.** [1, 2, 3]

b and a refer to the same list object, so changing b changes a.

</details>

---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Build the vocabulary dict

●○○ · ~6 min

> A vocabulary maps each distinct token to an integer id — the first step before any count vector or embedding lookup. Numbering in order of first appearance keeps ids stable and reproducible.

Implement `build_vocab(docs)` → dict mapping each distinct token to its index, numbered 0, 1, 2, … in order of first appearance across `docs` (a list of token lists).

<details><summary>Hint</summary>

One pass: `if t not in vocab: vocab[t] = len(vocab)`.

</details>


In [ ]:
import numpy as np

def build_vocab(docs):
    vocab = {}
    # your code — watch the ordering rule in the prompt
    return vocab

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    vocab = [\"alpha\", \"beta\", \"gamma\", \"delta\", \"eps\", \"zeta\"]\n    m = int(rng.integers(2, 6))\n    docs = [[str(w) for w in rng.choice(vocab, int(rng.integers(2, 7)))]\n            for _ in range(m)]\n    return {\"docs\": docs}\n\ndef build_vocab(docs):\n    vocab = {}\n    for d in docs:\n        for t in d:\n            if t not in vocab:\n                vocab[t] = len(vocab)\n    return vocab", "build_vocab", 20)

### Task 2 — Near-duplicate pair detection

●●● · ~45 min

> Near-duplicate detection comes up often in olympiad tasks: vectorize each document (raw token counts work), compute pairwise cosine similarity, and flag pairs above a threshold. Returning the pairs sorted keeps the output deterministic and diff-friendly.

Implement `dup_pairs(docs, threshold)` → sorted list of `(i, j)` index pairs with `i < j`, whose cosine similarity is `>= threshold`. `docs` is a list of token lists; a doc's vector is its token-count vector over the sorted vocabulary; a zero vector has similarity 0 with everything.

<details><summary>Hint</summary>

Build the count matrix V, normalize its rows with the zero-norm guard, S = Vn @ Vn.T, then scan all i < j.

</details>


In [ ]:
import numpy as np

def dup_pairs(docs, threshold):
    # your code
    return []

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    vocab = [\"aa\", \"bb\", \"cc\", \"dd\", \"ee\"]\n    m = int(rng.integers(3, 8))\n    docs = [[str(w) for w in rng.choice(vocab, int(rng.integers(2, 7)))]\n            for _ in range(m)]\n    if rng.random() < 0.15:\n        docs[int(rng.integers(m))] = []   # empty doc -> zero vector, sim 0 with all\n    if rng.random() < 0.6:          # inject a duplicate\n        i, j = rng.choice(m, 2, replace=False)\n        docs[int(j)] = list(docs[int(i)])\n    if rng.random() < 0.2:\n        # snap threshold onto a pair's realized cosine so \">=\" vs \">\" is decided\n        vocab_all = sorted({t for d in docs for t in d})\n        col = {t: k for k, t in enumerate(vocab_all)}\n        def vec(dd):\n            v = np.zeros(len(vocab_all))\n            for t in dd:\n                v[col[t]] += 1.0\n            return v\n        ii, jj = [int(x) for x in rng.choice(m, 2, replace=False)]\n        a, b = vec(docs[ii]), vec(docs[jj])\n        na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))\n        if na > 0 and nb > 0:\n            # match the reference's normalise-then-dot order so the snapped\n            # threshold lands bit-exactly on the realized cosine\n            return {\"docs\": docs, \"threshold\": float((a / na) @ (b / nb))}\n    return {\"docs\": docs,\n            \"threshold\": float(rng.choice([0.71, 0.77, 0.83, 0.94]))}\n\ndef dup_pairs(docs, threshold):\n    vocab = sorted({t for d in docs for t in d})\n    col = {t: i for i, t in enumerate(vocab)}\n    V = np.zeros((len(docs), len(vocab)))\n    for i, d in enumerate(docs):\n        for t in d:\n            V[i, col[t]] += 1.0\n    n = np.linalg.norm(V, axis=1, keepdims=True)\n    Vn = np.where(n == 0, 0.0, V / np.where(n == 0, 1.0, n))\n    S = Vn @ Vn.T\n    out = []\n    for i in range(len(docs)):\n        for j in range(i + 1, len(docs)):\n            if S[i, j] >= threshold:\n                out.append((i, j))\n    return out", "dup_pairs", 20)

### Task 3 — Token frequency counter

●○○ · ~8 min

> The simplest tokenizer splits text on whitespace. Counting token frequencies is the basis of Bag-of-Words, TF-IDF and vocab building. Case matters — 'The' and 'the' are the same word, so lowercase first.

Implement `token_counts(text)` → dict mapping each **lowercased** whitespace-separated token to its count.

<details><summary>Hint</summary>

Loop over text.lower().split() and use counts.get(w, 0) + 1 — or collections.Counter.

</details>


In [ ]:
import numpy as np

def token_counts(text):
    # your code
    return {}

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    vocab = [\"the\", \"a\", \"cat\", \"dog\", \"sits\", \"runs\", \"llm\", \"ai\", \"is\", \"on\"]\n    n = int(rng.integers(4, 20))\n    words = [str(w) for w in rng.choice(vocab, n)]\n    words = [w.upper() if rng.random() < 0.3 else w for w in words]\n    return {\"text\": \" \".join(words)}\n\ndef token_counts(text):\n    counts = {}\n    for w in text.lower().split():\n        counts[w] = counts.get(w, 0) + 1\n    return counts", "token_counts", 20)

---

**Go deeper**
- [The Python Tutorial (official)](https://docs.python.org/3/tutorial/)